# baseline model — nested logistic regression (sq2)

base = demographics + ses. expanded = base + healthcare access (insurance, personal doctor, cost barrier). compare with LR test + AIC.

first chunk re-runs the shared starter so it's standalone; jump to the SQ2 section if you've already loaded the split.

_Nick_

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

RANDOM_STATE = 42  # shared seed for reproducibility

KAGGLE_DIR = Path("/kaggle/input/datasets/ajenks/brfss-2020-2024-cleaned-and-weighted")
LOCAL_DIR = Path("cleaned")

if KAGGLE_DIR.exists():
    CLEAN_DIR = KAGGLE_DIR
    ENV = "Kaggle"
else:
    CLEAN_DIR = LOCAL_DIR
    ENV = "Local"

print(f"Environment: {ENV}")
print(f"Data directory: {CLEAN_DIR}")

In [ ]:
# Columns from the pooled file that all SQs may need
POOLED_COLS = [
    "RECENT_CHECKUP",       # 1 = checkup within past year
    "GOT_FLUSHOT",          # 1 = flu shot in past 12 months
    "HAS_EXERCISE",         # 1 = physical activity in past 30 days
    "_RFSMOK3",             # 1 = not a current smoker, 2 = current smoker

    "_AGE_G",               # age group (1-6)
    "_SEX",                 # 1 = male, 2 = female
    "_IMPRACE",             # imputed race/ethnicity (1-6)
    "MARITAL",              # marital status (1-6)

    "_EDUCAG",              # education level (1-4)
    "EMPLOY1",              # employment status (1-8)

    "_BMI5_SCALED",         # BMI (continuous, already divided by 100)
    "HAS_DEPRESSION",       # 1 = told had depressive disorder
    "ADDEPEV3",             # depression raw (for alternate encoding)

    "HAS_PERSONAL_DOCTOR",  # 1 = has personal doctor

    "STATE_FIPS",           # state FIPS code

    "sample_weight",        # normalized pooled weight for model fitting
    "SURVEY_YEAR",          # source year (2020-2024)
]

df = pd.read_parquet(CLEAN_DIR / "brfss_2020_2024_pooled_ml.parquet", columns=POOLED_COLS)

# Downcast floats to save memory
for col in df.select_dtypes(include=["float64"]).columns:
    df[col] = pd.to_numeric(df[col], downcast="float")

print(f"Pooled dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nYear distribution:")
print(df["SURVEY_YEAR"].value_counts().sort_index())

In [ ]:

income_parts = []

# 2020: _INCOMG already has the 5-level scheme
yr_df = pd.read_parquet(CLEAN_DIR / "brfss_2020_ml.parquet", columns=["_INCOMG"])
yr_df.rename(columns={"_INCOMG": "INCOME_GROUP"}, inplace=True)
income_parts.append(yr_df)

# 2021-2024: _INCOMG1 needs collapsing
for year in [2021, 2022, 2023, 2024]:
    yr_df = pd.read_parquet(CLEAN_DIR / f"brfss_{year}_ml.parquet", columns=["_INCOMG1"])
    # Collapse upper income brackets: 5,6,7 -> 5
    yr_df["INCOME_GROUP"] = yr_df["_INCOMG1"].where(yr_df["_INCOMG1"] <= 4, 5.0)
    yr_df.drop(columns=["_INCOMG1"], inplace=True)
    income_parts.append(yr_df)

income_col = pd.concat(income_parts, ignore_index=True)
df["INCOME_GROUP"] = income_col["INCOME_GROUP"].values
del income_parts, income_col

print("INCOME_GROUP value counts:")
print(df["INCOME_GROUP"].value_counts(dropna=False).sort_index())
print(f"\nLabels: 1=<$15k, 2=$15-25k, 3=$25-35k, 4=$35-50k, 5=$50k+")

In [ ]:

access_parts = []

for year in [2020, 2021, 2022, 2023, 2024]:
    yr_path = CLEAN_DIR / f"brfss_{year}_ml.parquet"

    # Determine which columns to load from this year
    load_cols = []
    if year in (2020, 2024):
        load_cols.append("HAS_HEALTH_PLAN")
    if year in (2021, 2022, 2023, 2024):
        load_cols.append("COST_BARRIER")

    # 2020 has MEDCOST (raw) instead of COST_BARRIER — derive it
    if year == 2020:
        load_cols.append("MEDCOST")

    yr_df = pd.read_parquet(yr_path, columns=load_cols)

    # Derive COST_BARRIER for 2020 from MEDCOST (1=Yes, 2=No, 7=DK, 9=Ref)
    if year == 2020:
        yr_df["MEDCOST"] = yr_df["MEDCOST"].replace({7.0: np.nan, 9.0: np.nan})
        yr_df["COST_BARRIER"] = np.where(
            yr_df["MEDCOST"].isna(), np.nan,
            np.where(yr_df["MEDCOST"] == 1.0, 1.0, 0.0),
        )
        yr_df.drop(columns=["MEDCOST"], inplace=True)

    # Fill missing columns with NaN (variable not asked that year)
    for col in ["HAS_HEALTH_PLAN", "COST_BARRIER"]:
        if col not in yr_df.columns:
            yr_df[col] = np.nan

    access_parts.append(yr_df[["HAS_HEALTH_PLAN", "COST_BARRIER"]])

access_df = pd.concat(access_parts, ignore_index=True)
df["HAS_HEALTH_PLAN"] = access_df["HAS_HEALTH_PLAN"].values
df["COST_BARRIER"] = access_df["COST_BARRIER"].values
del access_parts, access_df

df["HAS_HEALTH_PLAN_MISSING"] = df["HAS_HEALTH_PLAN"].isna().astype("float32")
df["COST_BARRIER_MISSING"] = df["COST_BARRIER"].isna().astype("float32")

print("HAS_HEALTH_PLAN availability by year:")
print(df.groupby("SURVEY_YEAR")["HAS_HEALTH_PLAN"].apply(lambda s: s.notna().mean()).round(3))
print("\nCOST_BARRIER availability by year:")
print(df.groupby("SURVEY_YEAR")["COST_BARRIER"].apply(lambda s: s.notna().mean()).round(3))

In [ ]:
df["NON_SMOKER"] = np.where(
    df["_RFSMOK3"].isna(), np.nan,
    np.where(df["_RFSMOK3"] == 1.0, 1.0, 0.0),
)

TARGET_COMPONENTS = ["RECENT_CHECKUP", "GOT_FLUSHOT", "HAS_EXERCISE", "NON_SMOKER"]

# Only compute the score where ALL four components are non-null
all_present = df[TARGET_COMPONENTS].notna().all(axis=1)
df["PREV_HEALTH_SCORE"] = np.where(
    all_present,
    df[TARGET_COMPONENTS].sum(axis=1),
    np.nan,
)

df["PREV_HEALTH_HIGH"] = np.where(
    df["PREV_HEALTH_SCORE"].isna(), np.nan,
    np.where(df["PREV_HEALTH_SCORE"] >= 3, 1.0, 0.0),
)

print("Composite score distribution:")
print(df["PREV_HEALTH_SCORE"].value_counts(dropna=False).sort_index())
print(f"\nBinary target (PREV_HEALTH_HIGH):")
print(df["PREV_HEALTH_HIGH"].value_counts(dropna=False).sort_index())
print(f"\nRows with valid target: {df['PREV_HEALTH_HIGH'].notna().sum():,} / {len(df):,}"
      f" ({df['PREV_HEALTH_HIGH'].notna().mean()*100:.1f}%)")

In [ ]:

DEMO_FEATURES = ["_AGE_G", "_SEX", "_IMPRACE", "MARITAL"]

SOCIO_FEATURES = ["_EDUCAG", "INCOME_GROUP", "EMPLOY1"]

BEHAVIORAL_FEATURES = ["_BMI5_SCALED", "HAS_DEPRESSION"]

ACCESS_FEATURES = [
    "HAS_PERSONAL_DOCTOR",
    "HAS_HEALTH_PLAN", "HAS_HEALTH_PLAN_MISSING",
    "COST_BARRIER", "COST_BARRIER_MISSING",
]

GEO_FEATURES = ["STATE_FIPS"]

# Combined (all available predictors)
ALL_FEATURES = DEMO_FEATURES + SOCIO_FEATURES + BEHAVIORAL_FEATURES + ACCESS_FEATURES + GEO_FEATURES

print("Feature groups:")
print(f"  Demographic:       {DEMO_FEATURES}")
print(f"  Socioeconomic:     {SOCIO_FEATURES}")
print(f"  Behavioral/Health: {BEHAVIORAL_FEATURES}")
print(f"  Healthcare Access: {ACCESS_FEATURES}")
print(f"  Geography:         {GEO_FEATURES}")
print(f"\n  Total predictors:  {len(ALL_FEATURES)}")

In [ ]:
df_model = df[df["PREV_HEALTH_HIGH"].notna()].copy()
print(f"Modeling partition: {len(df_model):,} rows (dropped {len(df) - len(df_model):,} with missing target)")

y = df_model["PREV_HEALTH_HIGH"]

train_idx, test_idx = train_test_split(
    df_model.index,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

df_train = df_model.loc[train_idx].copy()
df_test = df_model.loc[test_idx].copy()

print(f"\nTrain: {len(df_train):,}  |  Test: {len(df_test):,}")
print(f"\nTarget distribution (train):")
print(df_train["PREV_HEALTH_HIGH"].value_counts(normalize=True).sort_index().round(4))
print(f"\nTarget distribution (test):")
print(df_test["PREV_HEALTH_HIGH"].value_counts(normalize=True).sort_index().round(4))

In [ ]:
checks = []

# 1. No overlap between train and test indices
overlap = set(train_idx) & set(test_idx)
checks.append(("No train/test overlap", len(overlap) == 0))

# 2. Stratification preserved class ratios
train_ratio = df_train["PREV_HEALTH_HIGH"].mean()
test_ratio = df_test["PREV_HEALTH_HIGH"].mean()
checks.append(("Class ratio preserved (within 0.5pp)", abs(train_ratio - test_ratio) < 0.005))

# 3. Sample weights are present and positive
checks.append(("Train weights all positive", (df_train["sample_weight"] > 0).all()))
checks.append(("Test weights all positive", (df_test["sample_weight"] > 0).all()))

# 4. Split ratio is approximately 80/20
actual_test_pct = len(df_test) / (len(df_train) + len(df_test))
checks.append(("Split ratio ~20% test", abs(actual_test_pct - 0.20) < 0.01))

# 5. All five survey years present in both partitions
checks.append(("All years in train", df_train["SURVEY_YEAR"].nunique() == 5))
checks.append(("All years in test", df_test["SURVEY_YEAR"].nunique() == 5))

print("Verification:")
for label, passed in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {label}")

In [ ]:
print("Missing values in training set (key columns):\n")
cols_check = ALL_FEATURES + TARGET_COMPONENTS + ["PREV_HEALTH_SCORE", "PREV_HEALTH_HIGH"]
missing = df_train[cols_check].isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    for col, cnt in missing.items():
        print(f"  {col:30s}  {cnt:>8,}  ({cnt/len(df_train)*100:5.1f}%)")
else:
    print("  No missing values in key columns.")
print(f"\nTotal training rows: {len(df_train):,}")

In [ ]:

SQ1_COLS = [
    # Survey design
    "_PSU", "_STSTR", "_LLCPWT_POOLED",
    # Individual behaviors (targets for behavior-specific tests)
    "RECENT_CHECKUP", "GOT_FLUSHOT", "HAS_EXERCISE", "_RFSMOK3",
    # Grouping variables
    "_AGE_G", "_SEX", "_IMPRACE", "_EDUCAG", "EMPLOY1", "MARITAL",
    "STATE_FIPS", "SURVEY_YEAR",
]

df_sq1 = pd.read_parquet(CLEAN_DIR / "brfss_2020_2024_pooled_eda.parquet", columns=SQ1_COLS)

# Derive NON_SMOKER and composite for SQ1 as well
df_sq1["NON_SMOKER"] = np.where(
    df_sq1["_RFSMOK3"].isna(), np.nan,
    np.where(df_sq1["_RFSMOK3"] == 1.0, 1.0, 0.0),
)
comps = ["RECENT_CHECKUP", "GOT_FLUSHOT", "HAS_EXERCISE", "NON_SMOKER"]
all_ok = df_sq1[comps].notna().all(axis=1)
df_sq1["PREV_HEALTH_SCORE"] = np.where(all_ok, df_sq1[comps].sum(axis=1), np.nan)
df_sq1["PREV_HEALTH_HIGH"] = np.where(
    df_sq1["PREV_HEALTH_SCORE"].isna(), np.nan,
    np.where(df_sq1["PREV_HEALTH_SCORE"] >= 3, 1.0, 0.0),
)

df_sq1["INCOME_GROUP"] = df["INCOME_GROUP"].values

print(f"SQ1 dataset: {df_sq1.shape[0]:,} rows x {df_sq1.shape[1]} columns")
print(f"Design vars present: _PSU={df_sq1['_PSU'].notna().all()}, _STSTR={df_sq1['_STSTR'].notna().all()}")
print(f"\nExample: weighted prevalence of flu shot")
sub = df_sq1.dropna(subset=["GOT_FLUSHOT"])
wtd = (sub["GOT_FLUSHOT"] * sub["_LLCPWT_POOLED"]).sum() / sub["_LLCPWT_POOLED"].sum()
print(f"  Population-weighted flu shot rate: {wtd*100:.1f}%")

In [ ]:

# Base model predictors (demographic + socioeconomic)
SQ2_BASE_FEATURES = DEMO_FEATURES + SOCIO_FEATURES

# Expanded model adds healthcare access
SQ2_EXPANDED_FEATURES = SQ2_BASE_FEATURES + ACCESS_FEATURES

print("SQ2 Base model features:")
print(f"  {SQ2_BASE_FEATURES}")
print(f"\nSQ2 Expanded model adds:")
print(f"  {ACCESS_FEATURES}")

sq2_train = df_train.copy()
sq2_test = df_test.copy()

for col in ["HAS_HEALTH_PLAN", "COST_BARRIER"]:
    train_mode = sq2_train.loc[sq2_train[col].notna(), col].mode().iloc[0]
    sq2_train[col] = sq2_train[col].fillna(train_mode)
    sq2_test[col] = sq2_test[col].fillna(train_mode)
    print(f"\n{col}: imputed missing with mode = {train_mode:.0f}")
    print(f"  Train non-null: {sq2_train[col].notna().sum():,}")
    print(f"  Test non-null:  {sq2_test[col].notna().sum():,}")

print("\nThe _MISSING indicator columns are still present so the model can")
print("distinguish 'imputed because not asked' from actual responses.")

In [ ]:

# Full feature set for SQ3 (all predictor families compete)
SQ3_FEATURES = ALL_FEATURES + ["SURVEY_YEAR"]

# Separate X, y, weights for train and test
X_train_sq3 = df_train[SQ3_FEATURES].copy()
y_train_sq3 = df_train["PREV_HEALTH_HIGH"].copy()
w_train_sq3 = df_train["sample_weight"].copy()

X_test_sq3 = df_test[SQ3_FEATURES].copy()
y_test_sq3 = df_test["PREV_HEALTH_HIGH"].copy()
w_test_sq3 = df_test["sample_weight"].copy()

# Per-behavior targets for secondary analysis
BEHAVIOR_TARGETS = {
    "checkup":  "RECENT_CHECKUP",
    "flushot":  "GOT_FLUSHOT",
    "exercise": "HAS_EXERCISE",
    "nonsmoker": "NON_SMOKER",
}

print("SQ3 ready:")
print(f"  Features: {len(SQ3_FEATURES)}")
print(f"  Train: {X_train_sq3.shape}")
print(f"  Test:  {X_test_sq3.shape}")
print(f"\n  Per-behavior targets available: {list(BEHAVIOR_TARGETS.keys())}")
print(f"\nNext steps: feature encoding, imputation, model fitting, tuning.")

In [ ]:

# Branch 1: Demographic + Socioeconomic
NN_BRANCH1_FEATURES = DEMO_FEATURES + SOCIO_FEATURES + GEO_FEATURES

# Branch 2: Healthcare Access + Behavioral Context
NN_BRANCH2_FEATURES = ACCESS_FEATURES + BEHAVIORAL_FEATURES + ["SURVEY_YEAR"]

print("NN Branch 1 (Demo/Socio):")
print(f"  {NN_BRANCH1_FEATURES}")
print(f"\nNN Branch 2 (Access/Behavioral):")
print(f"  {NN_BRANCH2_FEATURES}")

MULTI_TARGETS = ["RECENT_CHECKUP", "GOT_FLUSHOT", "HAS_EXERCISE", "NON_SMOKER"]
y_train_multi = df_train[MULTI_TARGETS].copy()
y_test_multi = df_test[MULTI_TARGETS].copy()

print(f"\nMulti-output target shape (train): {y_train_multi.shape}")
print(f"Multi-output target shape (test):  {y_test_multi.shape}")
print(f"Multi-output missing per column (train):")
print(y_train_multi.isnull().sum())

In [ ]:
demosocio_train = df_train[NN_BRANCH1_FEATURES].copy()
accbehav_train = df_train[NN_BRANCH2_FEATURES].copy()
demosocio_test = df_test[NN_BRANCH1_FEATURES].copy()
accbehav_test = df_test[NN_BRANCH2_FEATURES].copy()

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

imputer = SimpleImputer(strategy = "most_frequent")
imputer1 = SimpleImputer(strategy = "most_frequent", add_indicator = True)
imputer2 = SimpleImputer(strategy = "median", add_indicator = True)

cols_to_impute = ['_AGE_G', '_SEX', '_IMPRACE', 'MARITAL', '_EDUCAG', 'INCOME_GROUP', 'EMPLOY1', 'STATE_FIPS']
imputed_demo_train = imputer1.fit_transform(demosocio_train[cols_to_impute])
imputed_demo_test = imputer1.transform(demosocio_test[cols_to_impute])
new_cols = imputer1.get_feature_names_out(cols_to_impute)

demosocio_train_imputed = pd.DataFrame(imputed_demo_train, columns = new_cols, index = demosocio_train.index)
demosocio_test_imputed = pd.DataFrame(imputed_demo_test, columns = new_cols, index = demosocio_test.index)

col_to_impute = ['HAS_HEALTH_PLAN','COST_BARRIER', 'HAS_PERSONAL_DOCTOR', 'HAS_DEPRESSION']
accbehav_train[col_to_impute] = imputer.fit_transform(accbehav_train[col_to_impute])
accbehav_test[col_to_impute] = imputer.transform(accbehav_test[col_to_impute])
bmi_imputed_train = imputer2.fit_transform(accbehav_train[["_BMI5_SCALED"]])
bmi_imputed_test = imputer2.transform(accbehav_test[["_BMI5_SCALED"]])
accbehav_train["_BMI5_SCALED"] = bmi_imputed_train[:, 0]
accbehav_train["_BMI5_MISSING"] = bmi_imputed_train[:, 1]
accbehav_test["_BMI5_SCALED"] = bmi_imputed_test[:, 0]
accbehav_test["_BMI5_MISSING"] = bmi_imputed_test[:, 1]

# Scale BMI
scaler = StandardScaler()
accbehav_train[["_BMI5_SCALED"]] = scaler.fit_transform(accbehav_train[["_BMI5_SCALED"]])
accbehav_test[["_BMI5_SCALED"]] = scaler.transform(accbehav_test[["_BMI5_SCALED"]])

encoder_demo = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
x_branch1_train = encoder_demo.fit_transform(demosocio_train_imputed).astype(np.float32)
x_branch1_test = encoder_demo.transform(demosocio_test_imputed).astype(np.float32)
demo_cols = encoder_demo.get_feature_names_out(demosocio_train_imputed.columns)

# One-hot encode SURVEY_YEAR in the access/behavioral branch
encoder_year = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
year_train_encoded = encoder_year.fit_transform(accbehav_train[["SURVEY_YEAR"]])
year_test_encoded = encoder_year.transform(accbehav_test[["SURVEY_YEAR"]])
year_cols = encoder_year.get_feature_names_out(["SURVEY_YEAR"])

branch2_numeric_cols = ["HAS_HEALTH_PLAN", "COST_BARRIER", "HAS_PERSONAL_DOCTOR",
                        "HAS_DEPRESSION", "_BMI5_SCALED", "_BMI5_MISSING",
                        "HAS_HEALTH_PLAN_MISSING", "COST_BARRIER_MISSING"]

x_branch2_train = np.hstack([accbehav_train[branch2_numeric_cols].values, year_train_encoded]).astype(np.float32)
x_branch2_test = np.hstack([accbehav_test[branch2_numeric_cols].values, year_test_encoded]) .astype(np.float32)

#Convert targets to float32 datatype
y_train = df_train["PREV_HEALTH_HIGH"].values.astype(np.float32)
y_test = df_test["PREV_HEALTH_HIGH"].values.astype(np.float32)
y_train_multi.values.astype(np.float32)
y_test_multi.values.astype(np.float32)

print("X_branch1_train:", x_branch1_train.shape, "NaN:", np.isnan(x_branch1_train).any())
print("X_branch1_test: ", x_branch1_test.shape,  "NaN:", np.isnan(x_branch1_test).any())
print("X_branch2_train:", x_branch2_train.shape, "NaN:", np.isnan(x_branch2_train).any())
print("X_branch2_test: ", x_branch2_test.shape,  "NaN:", np.isnan(x_branch2_test).any())
print("y_train:", y_train.shape, "NaN:", np.isnan(y_train).any())
print("y_test: ", y_test.shape,  "NaN:", np.isnan(y_test).any())
print("y_train_multi:", y_train_multi.shape, "NaN:", np.isnan(y_train_multi).any())
print("y_test_multi: ", y_test_multi.shape,  "NaN:", np.isnan(y_test_multi).any())

In [ ]:
'''from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Dense, BatchNormalization,Activation,Dropout,Concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC

#Defining keras model
input1 = Input(shape = (101,), name = 'demo_socio' )
input2 = Input(shape = (13,), name = 'acc_behav')

#Branch 1 Layer 1
x1 = Dense(64)(input1)
x1 = BatchNormalization()(x1)
x1 = Activation("relu")(x1)
x1 = Dropout(0.1)(x1)

#Branch 1 Layer 2
x1 = Dense(64)(x1)
x1 = BatchNormalization()(x1)
x1 = Activation("relu")(x1)
x1 = Dropout(0.1)(x1)

#Branch 2 Layer 1 
x2 = Dense(32)(input2)
x2 = BatchNormalization()(x2)
x2 = Activation("relu")(x2)
x2 = Dropout(0.1)(x2)

#Concat and shared head
x3 = Concatenate()([x1,x2])
x3 = Dense(64)(x3)
x3 = BatchNormalization()(x3)
x3 = Activation("relu")(x3)
x3 = Dropout(0.1)(x3)

#output layer
output = Dense(1, activation = 'sigmoid')(x3)

#Model
model = Model(inputs = [input1, input2], outputs = output)

#Compile model
model.compile(optimizer = Adam(learning_rate=1e-3), loss = 'binary_crossentropy', metrics=[AUC(name='auc'), 'accuracy'])

print(model.summary())'''

from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Dense, BatchNormalization, Activation, Dropout, Concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC

def build_dual_path(
    b1_input_dim=101,
    b2_input_dim=13,
    b1_width=128,
    b1_layers=2,
    b2_width=32,
    b2_layers=1,
    head_width=64,
    head_layers=2,
    dropout=0.2,
    learning_rate=3e-4,
):
    """
    Build a dual-path binary classifier for preventive health behavior.

    Branch 1: demographic + socioeconomic + geographic features
    Branch 2: healthcare access + behavioral features
    """

    b1_input = Input(shape=(b1_input_dim,), name="demo_socio")
    x1 = b1_input
    for _ in range(b1_layers):
        x1 = Dense(b1_width)(x1)
        x1 = BatchNormalization()(x1)
        x1 = Activation("relu")(x1)
        x1 = Dropout(dropout)(x1)

    b2_input = Input(shape=(b2_input_dim,), name="acc_behav")
    x2 = b2_input
    for _ in range(b2_layers):
        x2 = Dense(b2_width)(x2)
        x2 = BatchNormalization()(x2)
        x2 = Activation("relu")(x2)
        x2 = Dropout(dropout)(x2)

    merged = Concatenate()([x1, x2])
    h = merged
    for _ in range(head_layers):
        h = Dense(head_width)(h)
        h = BatchNormalization()(h)
        h = Activation("relu")(h)
        h = Dropout(dropout)(h)

    output = Dense(1, activation="sigmoid", name="preventive_health")(h)

    model = Model(inputs=[b1_input, b2_input], outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[AUC(name="auc"), "accuracy"],
    )

    return model

# Build the model with dropout=0.1 for the diagnostic run
model = build_dual_path(dropout=0.1)
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

#callbacks
early_stop = EarlyStopping(monitor = 'val_auc', mode = 'max', patience = 10, restore_best_weights = True, verbose = 1)
checkpoint = ModelCheckpoint(filepath = '/kaggle/working/model_weights.keras', monitor = 'val_auc', mode = 'max', save_best_only = True, verbose = 1)

callbacks = [early_stop, checkpoint]

In [ ]:
#fit model
history = model.fit({"demo_socio": x_branch1_train, "acc_behav": x_branch2_train}, y_train, sample_weight = df_train["sample_weight"].values.astype(np.float32), validation_split = .1 , epochs = 50, batch_size = 1024, callbacks = callbacks, verbose = 2 )

In [ ]:
if ENV == "Kaggle":
    OUT_DIR = Path("/kaggle/working")
else:
    OUT_DIR = CLEAN_DIR

np.save(OUT_DIR / "train_indices.npy", np.array(train_idx))
np.save(OUT_DIR / "test_indices.npy", np.array(test_idx))
print(f"Saved train indices: {len(train_idx):,}")
print(f"Saved test indices:  {len(test_idx):,}")

df_train.to_parquet(OUT_DIR / "brfss_train.parquet", index=True, engine="pyarrow")
df_test.to_parquet(OUT_DIR / "brfss_test.parquet", index=True, engine="pyarrow")
print(f"\nSaved: {OUT_DIR}/brfss_train.parquet ({len(df_train):,} rows)")
print(f"Saved: {OUT_DIR}/brfss_test.parquet  ({len(df_test):,} rows)")
print(f"\nColumns in saved files:")
print(f"  {list(df_train.columns)}")